## Imports, constants, version-safe prefit calibration

In [3]:
import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, joblib
import sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import (average_precision_score, brier_score_loss, log_loss,
                             roc_auc_score, confusion_matrix)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
PROCESSED_DIR = Path("../data/processed")
MODEL_DIR = PROCESSED_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DATE = "2026-08-01"
CALIB_FRACTION = 0.20          # tail of train reserved for the calibrator
np.random.seed(RANDOM_SEED)

try:
    from sklearn.frozen import FrozenEstimator
    HAS_FROZEN = True
except ImportError:
    HAS_FROZEN = False

def make_prefit_calibrator(estimator, method):
    """Calibrate an already-fitted estimator on held-out data.
    One model, one monotone map -> ranking metrics are invariant."""
    if HAS_FROZEN:
        return CalibratedClassifierCV(FrozenEstimator(estimator), method=method)
    return CalibratedClassifierCV(estimator, method=method, cv="prefit")

print("sklearn", sklearn.__version__, "| FrozenEstimator:", HAS_FROZEN)

sklearn 1.7.2 | FrozenEstimator: True


## Load, derive the column lists from the contract

In [4]:
train = pd.read_csv(PROCESSED_DIR / "03_train.csv", parse_dates=["created_at"])
test  = pd.read_csv(PROCESSED_DIR / "03_test.csv",  parse_dates=["created_at"])
know  = pd.read_csv(PROCESSED_DIR / "03_feature_knowability.csv")

NUMERIC     = know.loc[know.model_role == "numeric",     "feature_name"].tolist()
CATEGORICAL = know.loc[know.model_role == "categorical", "feature_name"].tolist()
FEATURES    = NUMERIC + CATEGORICAL
FORBIDDEN   = set(know.loc[know.knowability == "forbidden", "feature_name"])

assert not (set(FEATURES) & FORBIDDEN), "forbidden feature reached the model"
assert set(FEATURES).issubset(train.columns) and set(FEATURES).issubset(test.columns)
assert train.created_at.max() < pd.Timestamp(SPLIT_DATE) <= test.created_at.min()
assert train[FEATURES].isna().sum().sum() == 0 and test[FEATURES].isna().sum().sum() == 0

print(f"train {train.shape}  pos={train.is_disputed.sum()} ({train.is_disputed.mean():.4%})")
print(f"test  {test.shape}   pos={test.is_disputed.sum()} ({test.is_disputed.mean():.4%})")
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = {len(FEATURES)} features")

train (74731, 30)  pos=643 (0.8604%)
test  (45246, 30)   pos=407 (0.8995%)
23 numeric + 3 categorical = 26 features


## Three disjoint temporal blocks

In [5]:
train = train.sort_values("created_at").reset_index(drop=True)
CALIB_CUT = train.created_at.quantile(1 - CALIB_FRACTION)

fit_blk = train[train.created_at <  CALIB_CUT].reset_index(drop=True)
cal_blk = train[train.created_at >= CALIB_CUT].reset_index(drop=True)

INNER_CUT = fit_blk.created_at.quantile(0.75)
inner_fit = fit_blk[fit_blk.created_at <  INNER_CUT]
inner_cal = fit_blk[fit_blk.created_at >= INNER_CUT]

X_fit, y_fit = fit_blk[FEATURES], fit_blk.is_disputed.values
X_cal, y_cal = cal_blk[FEATURES], cal_blk.is_disputed.values
X_tr,  y_tr  = train[FEATURES],   train.is_disputed.values
X_te,  y_te  = test[FEATURES],    test.is_disputed.values

for n, d in [("FIT", fit_blk), ("CAL/VAL", cal_blk), ("TEST", test)]:
    print(f"{n:8s} {len(d):>6,} rows  {d.created_at.min().date()} -> {d.created_at.max().date()}"
          f"  pos={d.is_disputed.sum():>3}  rate={d.is_disputed.mean():.4%}")

FIT      59,784 rows  2026-01-01 -> 2026-06-24  pos=515  rate=0.8614%
CAL/VAL  14,947 rows  2026-06-24 -> 2026-07-31  pos=128  rate=0.8564%
TEST     45,246 rows  2026-08-01 -> 2026-10-31  pos=407  rate=0.8995%


## Metrics, paired bootstrap, reliability

In [6]:
def precision_at_k(y, p, k):
    n = max(1, int(round(len(p) * k)))
    return float(np.asarray(y)[np.argsort(-p)[:n]].mean())

def evaluate(y, p, name, threshold=0.5):
    y = np.asarray(y); pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(
        model=name, n=len(y), positives=int(y.sum()), base_rate=float(y.mean()),
        pr_auc=average_precision_score(y, p),
        precision_at_1pct=precision_at_k(y, p, .01),
        lift_at_1pct=precision_at_k(y, p, .01) / y.mean(),
        precision_at_5pct=precision_at_k(y, p, .05),
        recall_at_5pct=float(y[np.argsort(-p)[:int(len(p)*.05)]].sum() / y.sum()),
        brier=brier_score_loss(y, p),
        log_loss=log_loss(y, p, labels=[0, 1]),
        max_proba=float(p.max()),
        roc_auc_DIAGNOSTIC_ONLY=roc_auc_score(y, p),
        tp_at_0p5=int(tp), fp_at_0p5=int(fp), fn_at_0p5=int(fn))

def paired_bootstrap_ap(y, p_a, p_b, n_boot=500, seed=RANDOM_SEED):
    """Resample rows, recompute both models' PR-AUC on the same resample.
    Paired, so it isolates the model gap from sampling noise in the positives."""
    y = np.asarray(y); rng = np.random.default_rng(seed); diffs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum() < 5:
            continue
        diffs.append(average_precision_score(y[i], p_a[i]) -
                     average_precision_score(y[i], p_b[i]))
    d = np.array(diffs)
    return dict(mean_gap=d.mean(), ci_low=np.percentile(d, 2.5),
                ci_high=np.percentile(d, 97.5), p_a_better=float((d > 0).mean()))

def reliability_table(y, p, n_bins=10):
    df = pd.DataFrame({"p": p, "y": np.asarray(y)})
    df["bin"] = pd.qcut(df.p, n_bins, duplicates="drop", labels=False)
    t = (df.groupby("bin", observed=True)
           .agg(n=("y", "size"), predicted_mean=("p", "mean"), observed_rate=("y", "mean"))
           .reset_index())
    t["gap"] = t.observed_rate - t.predicted_mean
    ece = float((t.n / t.n.sum() * t.gap.abs()).sum())
    return t, ece

## Preprocessors and model factories

In [7]:
def linear_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale",  StandardScaler())]), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])

def tree_preprocessor():
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), NUMERIC),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL)])

TREE_CAT_IDX = [len(NUMERIC) + i for i in range(len(CATEGORICAL))]

def make_lr(C=1.0, class_weight=None):
    return Pipeline([("prep", linear_preprocessor()),
                     ("clf", LogisticRegression(C=C, class_weight=class_weight,
                                                max_iter=3000, random_state=RANDOM_SEED))])

def make_hgb(**kw):
    params = dict(max_iter=300, learning_rate=0.05, max_depth=6, class_weight=None,
                  categorical_features=TREE_CAT_IDX, random_state=RANDOM_SEED)
    params.update(kw)
    return Pipeline([("prep", tree_preprocessor()),
                     ("clf", HistGradientBoostingClassifier(**params))])

## Stage 1: fit exactly what the LLD prescribed

In [8]:
prescribed = {
    "LLD_baseline_LR_balanced": make_lr(class_weight="balanced"),
    "LLD_main_HGB_balanced":    make_hgb(class_weight="balanced"),
}
val_rows, val_probas = [], {}
for name, model in prescribed.items():
    model.fit(X_fit, y_fit)
    p = model.predict_proba(X_cal)[:, 1]
    val_probas[name] = p
    val_rows.append(evaluate(y_cal, p, name))

print(pd.DataFrame(val_rows)[["model","pr_auc","precision_at_1pct","brier","log_loss","max_proba"]]
        .to_string(index=False))

                   model   pr_auc  precision_at_1pct    brier  log_loss  max_proba
LLD_baseline_LR_balanced 0.054052           0.134228 0.141254  0.447924   0.988793
   LLD_main_HGB_balanced 0.045047           0.080537 0.114329  0.378077   0.885732


## Stage 2: escalation

In [9]:
t0 = time.time()

lr_rows = []
for C in [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]:
    for cw in [None, "balanced"]:
        m = make_lr(C=C, class_weight=cw).fit(X_fit, y_fit)
        p = m.predict_proba(X_cal)[:, 1]
        lr_rows.append(dict(C=C, class_weight=str(cw),
                            val_pr_auc=average_precision_score(y_cal, p),
                            val_brier=brier_score_loss(y_cal, p)))
lr_grid = pd.DataFrame(lr_rows).sort_values("val_pr_auc", ascending=False)
best_plain    = lr_grid[lr_grid.class_weight == "None"].iloc[0]
best_weighted = lr_grid[lr_grid.class_weight == "balanced"].iloc[0]
print(lr_grid.head(6).to_string(index=False))

space = dict(learning_rate=[0.02, 0.03, 0.05, 0.08, 0.12],
             max_leaf_nodes=[4, 8, 15, 31],
             min_samples_leaf=[50, 150, 400, 800],
             l2_regularization=[0.0, 1.0, 10.0, 50.0],
             max_features=[0.6, 0.8, 1.0])
rng, tscv = np.random.default_rng(RANDOM_SEED), TimeSeriesSplit(n_splits=4)

hgb_rows = []
for _ in range(16):
    cfg = {k: rng.choice(v).item() for k, v in space.items()}
    folds = []
    for a, b in tscv.split(X_fit):
        m = make_hgb(max_iter=500, max_depth=None, early_stopping=True,
                     n_iter_no_change=30, validation_fraction=0.15, **cfg)
        m.fit(X_fit.iloc[a], y_fit[a])
        folds.append(average_precision_score(y_fit[b], m.predict_proba(X_fit.iloc[b])[:, 1]))
    hgb_rows.append({**cfg, "cv_pr_auc": float(np.mean(folds)), "cv_std": float(np.std(folds))})

hgb_search = pd.DataFrame(hgb_rows).sort_values("cv_pr_auc", ascending=False)
print(hgb_search.head(5).to_string(index=False))
print(f"search took {time.time()-t0:.0f}s")

best_cfg = {k: hgb_search.iloc[0][k] for k in space}
best_cfg["max_leaf_nodes"]   = int(best_cfg["max_leaf_nodes"])
best_cfg["min_samples_leaf"] = int(best_cfg["min_samples_leaf"])

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
    C class_weight  val_pr_auc  val_brier
10.00     balanced    0.054076   0.141224
 3.00     balanced    0.054056   0.141222
 1.00     balanced    0.054052   0.141254
 0.10     balanced    0.054044   0.141476
 0.30     balanced    0.054004   0.141308
 0.03     balanced    0.053803   0.141871
 learning_rate  max_leaf_nodes  min_samples_leaf  l2_regularization  max_features  cv_pr_auc   cv_std
          0.12               4               400                0.0           0.6   0.070516 0.016345
          0.08               4               150                0.0           1.0   0.068255 0.016526
          0.08               8               800               10.0           0.8   0.067501 0.012941
          0.05               4                50               10.0           1.0   0.067313 0.011644
          0.12               8            

## Stage 3: the selection rule, written down before it's applied

In [10]:
candidates = {
    "LR_plain":    make_lr(C=float(best_plain.C), class_weight=None),
    "LR_weighted": make_lr(C=float(best_weighted.C), class_weight="balanced"),
    "HGB_tuned":   make_hgb(max_iter=500, max_depth=None, early_stopping=True,
                            n_iter_no_change=30, validation_fraction=0.15, **best_cfg),
}
for name, model in candidates.items():
    model.fit(X_fit, y_fit)
    p = model.predict_proba(X_cal)[:, 1]
    val_probas[name] = p
    val_rows.append(evaluate(y_cal, p, name))

val_board = pd.DataFrame(val_rows).sort_values("pr_auc", ascending=False).reset_index(drop=True)
print(val_board[["model","pr_auc","precision_at_1pct","brier","log_loss"]].to_string(index=False))

leader, tie_set = val_board.model.iloc[0], [val_board.model.iloc[0]]
for other in val_board.model.iloc[1:]:
    bt = paired_bootstrap_ap(y_cal, val_probas[leader], val_probas[other])
    separated = bt["ci_low"] > 0
    print(f"{leader} vs {other}: gap={bt['mean_gap']:+.4f} "
          f"CI[{bt['ci_low']:+.4f},{bt['ci_high']:+.4f}] "
          f"P(better)={bt['p_a_better']:.2f} -> {'SEPARATED' if separated else 'TIE'}")
    if not separated:
        tie_set.append(other)

CLASS_RANK = {"LR_plain": 0, "LR_weighted": 0, "LLD_baseline_LR_balanced": 0,
              "HGB_tuned": 1, "LLD_main_HGB_balanced": 1}
vb = val_board.set_index("model")
SELECTED = sorted(tie_set, key=lambda m: (CLASS_RANK[m], vb.loc[m, "log_loss"]))[0]
print("TIE SET:", tie_set, "\nSELECTED:", SELECTED)

                   model   pr_auc  precision_at_1pct    brier  log_loss
             LR_weighted 0.054076           0.134228 0.141224  0.447823
LLD_baseline_LR_balanced 0.054052           0.134228 0.141254  0.447924
                LR_plain 0.052567           0.093960 0.008335  0.043659
               HGB_tuned 0.047329           0.080537 0.008385  0.044547
   LLD_main_HGB_balanced 0.045047           0.080537 0.114329  0.378077
LR_weighted vs LLD_baseline_LR_balanced: gap=+0.0000 CI[-0.0002,+0.0002] P(better)=0.66 -> TIE
LR_weighted vs LR_plain: gap=+0.0017 CI[-0.0020,+0.0060] P(better)=0.81 -> TIE
LR_weighted vs HGB_tuned: gap=+0.0061 CI[-0.0084,+0.0170] P(better)=0.85 -> TIE
LR_weighted vs LLD_main_HGB_balanced: gap=+0.0079 CI[-0.0073,+0.0215] P(better)=0.86 -> TIE
TIE SET: ['LR_weighted', 'LLD_baseline_LR_balanced', 'LR_plain', 'HGB_tuned', 'LLD_main_HGB_balanced'] 
SELECTED: LR_plain


## Stage 4: choose the calibration method, honestly

In [11]:
proto = candidates.get(SELECTED, prescribed.get(SELECTED))
inner = clone(proto).fit(inner_fit[FEATURES], inner_fit.is_disputed.values)

cal_rows = []
p_un = inner.predict_proba(X_cal)[:, 1]
cal_rows.append(dict(method="none(raw)",
                     val_pr_auc=average_precision_score(y_cal, p_un),
                     val_brier=brier_score_loss(y_cal, p_un),
                     val_log_loss=log_loss(y_cal, p_un, labels=[0, 1])))
for meth in ["sigmoid", "isotonic"]:
    c = make_prefit_calibrator(inner, meth).fit(inner_cal[FEATURES], inner_cal.is_disputed.values)
    p = c.predict_proba(X_cal)[:, 1]
    cal_rows.append(dict(method=meth,
                         val_pr_auc=average_precision_score(y_cal, p),
                         val_brier=brier_score_loss(y_cal, p),
                         val_log_loss=log_loss(y_cal, p, labels=[0, 1])))

cal_board = pd.DataFrame(cal_rows).sort_values("val_log_loss")
print(cal_board.to_string(index=False))
CAL_METHOD = cal_board.method.iloc[0]
print("CAL_METHOD ->", CAL_METHOD)

   method  val_pr_auc  val_brier  val_log_loss
  sigmoid    0.048567   0.008351      0.044070
none(raw)    0.048567   0.008330      0.044111
 isotonic    0.043765   0.008367      0.048669
CAL_METHOD -> sigmoid


## Fit final artifacts, score both splits, single test read

In [12]:
baseline_model  = make_lr(class_weight="balanced").fit(X_fit, y_fit)
main_model      = candidates["HGB_tuned"]
selected_model  = candidates.get(SELECTED, prescribed.get(SELECTED))
calibrated_model = (selected_model if CAL_METHOD == "none(raw)"
                    else make_prefit_calibrator(selected_model, CAL_METHOD).fit(X_cal, y_cal))

def score_frame(df):
    out = df[["payment_id", "created_at"]].copy()
    out["is_disputed"]       = df.is_disputed.values
    out["amount"]            = df.amount.values          # notebook 5's economics needs rupees
    out["merchant_category"] = df.merchant_category.values
    out["method"]            = df.method.values
    X = df[FEATURES]
    out["proba_baseline"]   = baseline_model.predict_proba(X)[:, 1]
    out["proba_main"]       = main_model.predict_proba(X)[:, 1]
    out["proba_selected"]   = selected_model.predict_proba(X)[:, 1]
    out["proba_calibrated"] = calibrated_model.predict_proba(X)[:, 1]
    return out

train_pred, test_pred = score_frame(train), score_frame(test)

test_board = pd.DataFrame([evaluate(y_te, test_pred[c], c) for c in
                           ["proba_baseline","proba_main","proba_selected","proba_calibrated"]])
print(test_board[["model","pr_auc","lift_at_1pct","precision_at_1pct",
                  "precision_at_5pct","brier","log_loss","max_proba"]].to_string(index=False))

bt = paired_bootstrap_ap(y_te, test_pred.proba_main.values, test_pred.proba_selected.values)
print("TEST main-vs-selected:", {k: round(v, 4) for k, v in bt.items()})
assert not np.allclose(test_pred.proba_calibrated, test_pred.proba_main)

           model   pr_auc  lift_at_1pct  precision_at_1pct  precision_at_5pct    brier  log_loss  max_proba
  proba_baseline 0.084182     17.216521           0.154867           0.076923 0.140433  0.445246   0.991342
      proba_main 0.081233     15.494869           0.139381           0.072060 0.008583  0.043598   0.398763
  proba_selected 0.087248     16.232719           0.146018           0.077807 0.008535  0.042664   0.474006
proba_calibrated 0.087248     16.232719           0.146018           0.077807 0.008567  0.042857   0.334371
TEST main-vs-selected: {'mean_gap': np.float64(-0.006), 'ci_low': np.float64(-0.0168), 'ci_high': np.float64(0.006), 'p_a_better': 0.13}


## Permutation importance (the .feature_importances_ fix)feature importances (BLK-003)

In [13]:
pi = permutation_importance(selected_model, X_cal, y_cal, scoring="average_precision",
                            n_repeats=8, random_state=RANDOM_SEED, n_jobs=2)
imp = (pd.DataFrame({"feature": FEATURES,
                     "importance": pi.importances_mean,
                     "importance_std": pi.importances_std})
         .sort_values("importance", ascending=False).reset_index(drop=True))
imp["rank"] = np.arange(1, len(imp) + 1)
imp["informative"] = imp.importance > 2 * imp.importance_std   # crude 2-sigma screen
print(imp.head(12).to_string(index=False))
print("zero/negative importance:", (imp.importance <= 0).sum(), "of", len(imp))

coef = None
if isinstance(selected_model.named_steps["clf"], LogisticRegression):
    names = selected_model.named_steps["prep"].get_feature_names_out()
    coef = pd.DataFrame({"term": names, "coefficient": selected_model.named_steps["clf"].coef_[0]})
    coef["odds_ratio"] = np.exp(coef.coefficient)
    coef = coef.reindex(coef.coefficient.abs().sort_values(ascending=False).index).reset_index(drop=True)
    print(coef.head(10).to_string(index=False))

                     feature  importance  importance_std  rank  informative
                  log_amount    0.030750        0.003893     1         True
           email_domain_type    0.022946        0.002824     2         True
              phone_verified    0.022429        0.003116     3         True
           merchant_category    0.012961        0.001829     4         True
                      method    0.010973        0.001745     5         True
           has_prior_history    0.003068        0.001004     6         True
amount_vs_merchant_avg_ratio    0.002649        0.000929     7         True
    delivery_sla_days_filled    0.002481        0.001050     8         True
     is_first_txn_for_device    0.000981        0.000590     9        False
              is_foreign_bin    0.000820        0.001379    10        False
     account_age_days_at_txn    0.000748        0.001170    11        False
           is_physical_goods    0.000562        0.001021    12        False
zero/negativ

## Reliability, and the one place the model is wrong

In [14]:
rel_cal,  ece_cal  = reliability_table(y_te, test_pred.proba_calibrated.values)
rel_main, ece_main = reliability_table(y_te, test_pred.proba_main.values)
rel = pd.concat([rel_cal.assign(model="calibrated"), rel_main.assign(model="main_hgb")])
print(rel_cal.to_string(index=False))
print(f"ECE calibrated={ece_cal:.5f}  main={ece_main:.5f}")

top = test_pred.nlargest(int(0.05 * len(test_pred)), "proba_calibrated")
print(f"top-5% slice: predicted {top.proba_calibrated.mean():.4%} vs observed {top.is_disputed.mean():.4%}")

 bin    n  predicted_mean  observed_rate       gap
   0 4525        0.001075       0.000663 -0.000412
   1 4525        0.001724       0.001105 -0.000619
   2 4524        0.002278       0.002431  0.000153
   3 4525        0.002905       0.003315  0.000410
   4 4524        0.003696       0.003316 -0.000380
   5 4525        0.004710       0.003094 -0.001616
   6 4524        0.006203       0.003979 -0.002224
   7 4525        0.008627       0.007735 -0.000892
   8 4524        0.013723       0.013484 -0.000239
   9 4525        0.039295       0.050829  0.011534
ECE calibrated=0.00185  main=0.00137
top-5% slice: predicted 5.5234% vs observed 7.7807%


## Export with round-trip asserts

In [15]:
def export(df, name):
    path = PROCESSED_DIR / name
    df.to_csv(path, index=False)
    chk = pd.read_csv(path)
    assert chk.shape[0] == df.shape[0], f"row mismatch on {name}"
    assert set(chk.columns) == set(df.columns), f"column mismatch on {name}"
    print(f"OK {name} — {chk.shape[0]} rows, {chk.shape[1]} cols")

export(train_pred, "04_train_predictions.csv")
export(test_pred,  "04_test_predictions.csv")
export(imp,        "04_feature_importances.csv")
export(test_board, "04_test_metrics.csv")
export(val_board,  "04_validation_leaderboard.csv")
export(hgb_search, "04_hgb_search_results.csv")
export(rel,        "04_reliability_bins.csv")
if coef is not None:
    export(coef,   "04_model_coefficients.csv")

export(pd.DataFrame({"feature_name": FEATURES,
                     "model_role": ["numeric"]*len(NUMERIC) + ["categorical"]*len(CATEGORICAL),
                     "order": range(len(FEATURES))}), "04_feature_columns.csv")

export(pd.DataFrame([dict(key="selected_model",     value=SELECTED),
                     dict(key="calibration_method", value=CAL_METHOD),
                     dict(key="calibration_cut",    value=str(CALIB_CUT)),
                     dict(key="split_date",         value=SPLIT_DATE),
                     dict(key="random_seed",        value=RANDOM_SEED),
                     dict(key="sklearn_version",    value=sklearn.__version__),
                     dict(key="n_features",         value=len(FEATURES))]), "04_model_card.csv")

for obj, fn in [(baseline_model, "baseline_model.joblib"),
                (main_model,     "main_model.joblib"),
                (calibrated_model,"calibrated_model.joblib")]:
    joblib.dump(obj, MODEL_DIR / fn)

one = test[FEATURES].iloc[[0]]
for fn in ["baseline_model.joblib", "main_model.joblib", "calibrated_model.joblib"]:
    p = joblib.load(MODEL_DIR / fn).predict_proba(one)[0, 1]
    assert 0 <= p <= 1
    print(f"reload OK {fn} -> {p:.5f}")

OK 04_train_predictions.csv — 74731 rows, 10 cols
OK 04_test_predictions.csv — 45246 rows, 10 cols
OK 04_feature_importances.csv — 26 rows, 5 cols
OK 04_test_metrics.csv — 4 rows, 16 cols
OK 04_validation_leaderboard.csv — 5 rows, 16 cols
OK 04_hgb_search_results.csv — 16 rows, 7 cols
OK 04_reliability_bins.csv — 20 rows, 6 cols
OK 04_model_coefficients.csv — 38 rows, 3 cols
OK 04_feature_columns.csv — 26 rows, 3 cols
OK 04_model_card.csv — 7 rows, 2 cols
reload OK baseline_model.joblib -> 0.35923
reload OK main_model.joblib -> 0.00338
reload OK calibrated_model.joblib -> 0.00775


## Definition of done

In [16]:
tb = test_board.set_index("model")
checks = {
    "test_predictions row count == 03_test.csv":
        len(test_pred) == len(test),
    "proba_calibrated in [0,1]":
        bool(test_pred.proba_calibrated.between(0, 1).all()),
    "proba_calibrated != proba_main (calibration ran)":
        not np.allclose(test_pred.proba_calibrated, test_pred.proba_main),
    "deployed model lift@1% > 3x base rate":
        float(tb.loc["proba_calibrated", "lift_at_1pct"]) > 3,
    "deployed Brier <= prescribed-baseline Brier":
        float(tb.loc["proba_calibrated", "brier"]) <= float(tb.loc["proba_baseline", "brier"]),
    "ranking preserved through calibration":
        np.isclose(tb.loc["proba_calibrated","pr_auc"], tb.loc["proba_selected","pr_auc"]),
    "no forbidden feature used":
        not (set(FEATURES) & FORBIDDEN),
}
for k, v in checks.items():
    print(("PASS  " if v else "FAIL  ") + k)
assert all(checks.values())
print("\nSELECTED:", SELECTED, "| CALIBRATION:", CAL_METHOD)

PASS  test_predictions row count == 03_test.csv
PASS  proba_calibrated in [0,1]
PASS  proba_calibrated != proba_main (calibration ran)
PASS  deployed model lift@1% > 3x base rate
PASS  deployed Brier <= prescribed-baseline Brier
PASS  ranking preserved through calibration
PASS  no forbidden feature used

SELECTED: LR_plain | CALIBRATION: sigmoid
